In [1]:
import sys
import os
%load_ext autoreload
%autoreload 2
SCRIPT_DIR = os.path.dirname(os.path.realpath(__vsc_ipynb_file__))
sys.path.append(os.path.dirname(SCRIPT_DIR))

from pathlib import Path
import imageio
import json
import numpy as np
import pandas as pd



with open('../../assets/block_states.txt', 'r') as f:
    blocks = set()
    while line := f.readline():
        blocks.add(line.split('[')[0].split(':')[-1].strip())

blocks = list(sorted(blocks))
print(sorted(blocks))


['activator_rail', 'air', 'amethyst_block', 'amethyst_cluster', 'ancient_debris', 'andesite', 'anvil', 'azalea', 'bamboo', 'bamboo_mosaic', 'bamboo_sapling', 'banner', 'barrel', 'barrier', 'bars', 'basalt', 'beacon', 'bed', 'bedrock', 'bee_nest', 'beehive', 'beetroots', 'bell', 'big_dripleaf', 'big_dripleaf_stem', 'blackstone', 'blast_furnace', 'blue_ice', 'bone_block', 'bookshelf', 'brewing_stand', 'brick_block', 'brown_mushroom', 'brown_mushroom_block', 'bubble_column', 'budding_amethyst', 'button', 'cactus', 'cake', 'calcite', 'calibrated_sculk_sensor', 'campfire', 'candle', 'candle_cake', 'carpet', 'carrots', 'cartography_table', 'carved_pumpkin', 'cauldron', 'cave_air', 'cave_vines', 'cave_vines_plant', 'chain', 'chest', 'chiseled_bookshelf', 'chiseled_deepslate', 'chiseled_nether_bricks', 'chiseled_polished_blackstone', 'chiseled_quartz_block', 'chiseled_resin_bricks', 'chiseled_tuff', 'chiseled_tuff_bricks', 'chorus_flower', 'chorus_plant', 'clay', 'coal_block', 'coal_ore', 'coa

In [2]:
blocks = []
with open('../../assets/block_states.txt', 'r') as f:
    while line := f.readline():
        blocks.append(line.strip())

    

In [3]:
block_type2rgb = json.load(open('../../assets/block_type2rgb.json', 'r'))

In [4]:
import re

def get_base_name(block_str):
    match = re.search(r':([^\[\{\s]+)', block_str)
    return match.group(1) if match else block_str


df = pd.DataFrame({
    "block": blocks,
    "rgb": [block_type2rgb[get_base_name(block)] for block in blocks]
})

df["block"] = df["block"]
df["rgb"] = df["rgb"]

KeyError: 'portal'

In [ ]:
df.to_csv("../../assets/block_state2rgb.csv", index=False)


In [ ]:
print(pd.read_csv('../../assets/block_state2rgb.csv'))

                                                  block          rgb
0                               universal_minecraft:air        0|0|0
1       universal_minecraft:bedrock[infiniburn="false"]     46|46|46
2                             universal_minecraft:stone  125|125|125
3         universal_minecraft:diorite[polished="false"]  188|188|189
4                            universal_minecraft:gravel  126|124|122
...                                                 ...          ...
9751             minecraft:dark_oak_fence[block_data=2]     66|43|20
9752             minecraft:dark_oak_fence[block_data=3]     66|43|20
9753             minecraft:dark_oak_fence[block_data=4]     66|43|20
9754              minecraft:spruce_fence[block_data=10]    114|84|56
9755  universal_minecraft:redstone_wire[east="none",...      255|0|0

[9756 rows x 2 columns]


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from scipy.spatial.distance import pdist, squareform

# --- 1. Physics Parameters ---
ITERATIONS = 500       # How many "ticks" of gravity/repulsion
REPULSION = 30       # Strength of points pushing away
TETHER = 0.01          # Strength of points pulling back to original color
DAMPING = 0.9         # Velocity decay to prevent exploding

# --- 2. Data Preparation ---
# (Using your df logic)
orig_pos = np.array(df['rgb'].map(lambda x: [int(val) for val in x.split("|")]).tolist(), dtype=float)
current_pos = orig_pos.copy()# + np.random.normal(0, 1, orig_pos.shape)
velocity = np.zeros_like(current_pos)

# --- 3. Simulation Loop ---
print("Simulating gravity and repulsion...")
# for i in range(ITERATIONS):
#     # Calculate pairwise distances (expensive for 9k rows, so we use a subset or small radius)
#     # To keep it fast for 9755 rows, we'll use a simplified jitter-repulsion:
    
#     # Force 1: Repulsion (Push away from neighbors)
#     # Using a random sample for speed if df is huge
#     repulsion_force = np.zeros_like(current_pos)
#     idx = np.random.choice(len(current_pos), size=min(1000, len(current_pos)), replace=False)
    
#     # Force 2: Tether (Gravity toward original color)
#     tether_force = (orig_pos - current_pos) * TETHER
    
#     # Update Velocity and Position
#     velocity = (velocity + tether_force) * DAMPING
#     current_pos += velocity + (np.random.normal(0, REPULSION, current_pos.shape))
    
#     # Keep within RGB bounds [0, 255]
#     current_pos = np.clip(current_pos, 0, 255)

# Update DataFrame
rgb_df = pd.DataFrame(current_pos, columns=['R', 'G', 'B'])

# --- 4. Plotting ---
# Enable interactive widgets in Jupyter
%matplotlib qt
 

import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D

# Ensure interactive mode is on
plt.ion() 

fig = plt.figure(figsize=(10, 7))
ax = fig.add_subplot(111, projection='3d')

# Normalized RGB colors (Opaque as requested)
colors_normalized = orig_pos / 255.0

scatter = ax.scatter(
    rgb_df['R'], rgb_df['G'], rgb_df['B'],
    c=colors_normalized, 
    s=15,               # Adjusted size for performance
    alpha=1.0,          # Full opacity for RGB focus
    edgecolors='none'
)

# Axis Styling
ax.set_xlabel('Red')
ax.set_ylabel('Green')
ax.set_zlabel('Blue')
ax.set_title('Interactive Minecraft RGB Space')

# Set limits to the standard RGB range
ax.set_xlim(0, 255)
ax.set_ylim(0, 255)
ax.set_zlim(0, 255)

# Optional: Add a subtle grid background for depth perception
ax.xaxis.pane.fill = False
ax.yaxis.pane.fill = False
ax.zaxis.pane.fill = False

plt.tight_layout()
plt.show()

Simulating gravity and repulsion...
